In [ ]:
import sys
from pathlib import Path

# Ajouter src au path
sys.path.insert(0, str(Path.cwd().parent))

# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import warnings
warnings.filterwarnings('ignore')

# Imports du projet
from src import (
    DatasetGenerator,
    ManeuverPredictor,
    compare_models,
    create_circular_orbit,
    apply_maneuver
)

print("✓ Toutes les bibliothèques importées avec succès!")

## Part 1: Configuration et Paramètres

Configurons les paramètres pour notre démonstration.
- **Altitude orbitale**: 400 km (orbite basse)
- **Nombre de scénarios**: 500 (pour une démonstration rapide)
- **Δv maximum**: 0.050 km/s

In [ ]:
# Configuration des paramètres
ALTITUDE_RANGE = (380, 420)  # km
ALPHA = 1.0  # Poids carburant
BETA = 1e4   # Poids risque collision
MAX_DELTA_V = 0.050  # km/s
N_SCENARIOS_DEMO = 500  # Pour la démo
RANDOM_SEED = 42

print("Configuration:")
print(f"  Altitude: {ALTITUDE_RANGE} km")
print(f"  Δv maximum: {MAX_DELTA_V} km/s")
print(f"  Scénarios pour la démo: {N_SCENARIOS_DEMO}")
print(f"  Graine aléatoire: {RANDOM_SEED}")

## Part 2: Génération du Dataset

Générons un dataset avec des scénarios aléatoires d'évitement orbital.
Chaque scénario contient:
- État du satellite
- État du débris spatial
- Manœuvre optimale (obtenue par optimisation classique)

In [ ]:
print("=" * 60)
print("GÉNÉRATION DU DATASET")
print("=" * 60)

# Créer le générateur
generator = DatasetGenerator(
    altitude_range=ALTITUDE_RANGE,
    alpha=ALPHA,
    beta=BETA,
    max_delta_v=MAX_DELTA_V,
    random_seed=RANDOM_SEED
)

# Générer le dataset
print(f"\nGénération de {N_SCENARIOS_DEMO} scénarios...")
dataset = generator.generate_dataset(
    n_scenarios=N_SCENARIOS_DEMO,
    verbose=False
)

X = dataset['X']
y = dataset['y']

print(f"✓ Dataset généré!")
print(f"  Shape X (features): {X.shape}")
print(f"  Shape y (labels/manœuvres): {y.shape}")
print(f"\nFeatures (état du système):")
print(f"  - Position satellite (3D): colonnes 0-2")
print(f"  - Vitesse satellite (3D): colonnes 3-5")
print(f"  - Position débris (3D): colonnes 6-8")
print(f"  - Vitesse débris (3D): colonnes 9-11")
print(f"\nLabels (manœuvre optimale):")
print(f"  - Δv satellite (3D): colonnes 0-2")

## Part 3: Statistiques des Données

Analysons les statistiques du dataset généré.

In [ ]:
# Statistiques des features (entrées)
print("Statistiques des FEATURES (État du système):")
print("-" * 60)
features = ["Pos_Sat_X", "Pos_Sat_Y", "Pos_Sat_Z", 
            "Vel_Sat_X", "Vel_Sat_Y", "Vel_Sat_Z",
            "Pos_Deb_X", "Pos_Deb_Y", "Pos_Deb_Z",
            "Vel_Deb_X", "Vel_Deb_Y", "Vel_Deb_Z"]

stats_df = pd.DataFrame({
    'Feature': features,
    'Min': X.min(axis=0),
    'Max': X.max(axis=0),
    'Mean': X.mean(axis=0),
    'Std': X.std(axis=0)
})

print(stats_df.to_string(index=False))

# Statistiques des labels (sorties)
print("\n\nStatistiques des LABELS (Manœuvres Δv optimales):")
print("-" * 60)
labels = ["Δv_X", "Δv_Y", "Δv_Z"]

stats_labels = pd.DataFrame({
    'Label': labels,
    'Min': y.min(axis=0),
    'Max': y.max(axis=0),
    'Mean': y.mean(axis=0),
    'Std': y.std(axis=0)
})

print(stats_labels.to_string(index=False))

# Magnitude des manœuvres
delta_v_magnitude = np.linalg.norm(y, axis=1)
print(f"\nMagnitude des Δv:")
print(f"  Min: {delta_v_magnitude.min():.6f} km/s")
print(f"  Max: {delta_v_magnitude.max():.6f} km/s")
print(f"  Mean: {delta_v_magnitude.mean():.6f} km/s")
print(f"  Median: {np.median(delta_v_magnitude):.6f} km/s")

## Part 4: Visualisation des Données

Visualisons la distribution des manœuvres optimales.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Distribution des magnitudes de Δv
ax = axes[0, 0]
ax.hist(delta_v_magnitude * 1000, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Magnitude Δv (m/s)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des Magnitudes de Manœuvre Optimale')
ax.grid(True, alpha=0.3)

# Distribution des composantes Δv
ax = axes[0, 1]
ax.boxplot([y[:, 0] * 1000, y[:, 1] * 1000, y[:, 2] * 1000], 
           labels=['Δv_X', 'Δv_Y', 'Δv_Z'])
ax.set_ylabel('Valeur (m/s)')
ax.set_title('Distribution des Composantes Δv')
ax.grid(True, alpha=0.3, axis='y')

# Corrélation entre composantes
ax = axes[1, 0]
correlation = np.corrcoef(y.T)
im = ax.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['Δv_X', 'Δv_Y', 'Δv_Z'])
ax.set_yticklabels(['Δv_X', 'Δv_Y', 'Δv_Z'])
ax.set_title('Corrélation entre Composantes Δv')
plt.colorbar(im, ax=ax)

# Scatter plot Δv_X vs Δv_Y
ax = axes[1, 1]
scatter = ax.scatter(y[:, 0] * 1000, y[:, 1] * 1000, 
                     c=delta_v_magnitude * 1000, cmap='viridis', alpha=0.6, s=30)
ax.set_xlabel('Δv_X (m/s)')
ax.set_ylabel('Δv_Y (m/s)')
ax.set_title('Δv_X vs Δv_Y (coloré par magnitude)')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Magnitude (m/s)')

plt.tight_layout()
plt.show()

print("✓ Visualisations des données générées")

## Part 5: Entraînement et Comparaison des Modèles

Comparons 3 approches de machine learning:
- **Random Forest**: Modèle d'ensemble robuste
- **Gradient Boosting**: Modèle d'ensemble avec boosting
- **MLP**: Réseau de neurones multicouche

In [ ]:
print("=" * 60)
print("COMPARAISON DES MODÈLES")
print("=" * 60)
print("\nEntraînement des 3 modèles (esto peut prendre ~30-60s)...\n")

# Appel de la fonction de comparaison
comparison_results = compare_models(dataset, test_size=0.2)

print("\n" + "=" * 60)
print("RÉSULTATS DE COMPARAISON")
print("=" * 60)
print(comparison_results.to_string(index=False))

# Identifier le meilleur modèle
best_idx = comparison_results['test_r2'].idxmax()
best_model_type = comparison_results.loc[best_idx, 'model']
best_r2 = comparison_results.loc[best_idx, 'test_r2']

print(f"\n✓ Meilleur modèle: {best_model_type} (R² = {best_r2:.4f})")

## Part 6: Visualisation de la Comparaison

Graphiques comparatifs des performances des modèles.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Train R²
ax = axes[0]
colors = ['green' if model == best_model_type else 'steelblue' 
          for model in comparison_results['model']]
bars1 = ax.bar(comparison_results['model'], comparison_results['train_r2'], 
               color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('R² Score')
ax.set_title('Train R² Score')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis='y')
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Test R²
ax = axes[1]
colors = ['green' if model == best_model_type else 'steelblue' 
          for model in comparison_results['model']]
bars2 = ax.bar(comparison_results['model'], comparison_results['test_r2'], 
               color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('R² Score')
ax.set_title('Test R² Score')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis='y')
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom', fontsize=9)

# Plot 3: MAE
ax = axes[2]
mae_values = comparison_results['test_mae'].values * 1000  # Convertir en m/s
colors = ['green' if model == best_model_type else 'steelblue' 
          for model in comparison_results['model']]
bars3 = ax.bar(comparison_results['model'], mae_values, 
               color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('MAE (m/s)')
ax.set_title('Mean Absolute Error (Test Set)')
ax.grid(True, alpha=0.3, axis='y')
for bar in bars3:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("✓ Graphiques de comparaison générés")

## Part 7: Entrainement du Meilleur Modèle

Entraînons le meilleur modèle sur l'ensemble du dataset.

In [ ]:
from sklearn.model_selection import train_test_split

print("=" * 60)
print(f"ENTRAÎNEMENT DU MODÈLE: {best_model_type.upper()}")
print("=" * 60)

# Diviser les données
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nDivision des données:")
print(f"  Train: {X_train.shape[0]} samples")
print(f"  Test: {X_test.shape[0]} samples")

# Créer et entraîner le modèle
print(f"\nEntraînement en cours...")
best_model = ManeuverPredictor(model_type=best_model_type)
best_model.train(X_train, y_train, X_test, y_test)

# Faire des prédictions
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Calculer les métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\n✓ Entraînement terminé!")
print(f"\nMétriques Train:")
print(f"  MAE:  {train_mae*1000:.4f} m/s")
print(f"  RMSE: {train_rmse*1000:.4f} m/s")
print(f"  R²:   {train_r2:.4f}")

print(f"\nMétriques Test:")
print(f"  MAE:  {test_mae*1000:.4f} m/s")
print(f"  RMSE: {test_rmse*1000:.4f} m/s")
print(f"  R²:   {test_r2:.4f}")

## Part 8: Analyse des Prédictions

Visualisons les prédictions du modèle vs les valeurs réelles.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Calcul des magnitudes
y_test_mag = np.linalg.norm(y_test, axis=1) * 1000
y_test_pred_mag = np.linalg.norm(y_test_pred, axis=1) * 1000
residuals_mag = y_test_mag - y_test_pred_mag

# Row 1: Prédictions vs Réalité pour chaque composante
for i, (ax, label) in enumerate(zip(axes[0], ['ΔV_X', 'ΔV_Y', 'ΔV_Z'])):
    y_actual = y_test[:, i] * 1000
    y_pred = y_test_pred[:, i] * 1000
    
    ax.scatter(y_actual, y_pred, alpha=0.5, s=20)
    
    # Ligne de perfection
    lims = [min(y_actual.min(), y_pred.min()), max(y_actual.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
    
    ax.set_xlabel(f'Réalité {label} (m/s)')
    ax.set_ylabel(f'Prédiction {label} (m/s)')
    ax.set_title(f'Comparaison {label}')
    ax.grid(True, alpha=0.3)
    ax.legend()

# Row 2: Résidus et histogramme
ax = axes[1, 0]
ax.scatter(np.arange(len(residuals_mag)), residuals_mag, alpha=0.6, s=20, color='purple')
ax.axhline(y=0, color='r', linestyle='--', lw=2)
ax.set_xlabel('Index')
ax.set_ylabel('Résidu (m/s)')
ax.set_title('Résidus de la Magnitude')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.hist(residuals_mag, bins=30, color='purple', alpha=0.7, edgecolor='black')
ax.axvline(x=0, color='r', linestyle='--', lw=2)
ax.set_xlabel('Résidu (m/s)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des Résidus')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1, 2]
ax.scatter(y_test_mag, y_test_pred_mag, alpha=0.5, s=20, color='darkgreen')
lims = [min(y_test_mag.min(), y_test_pred_mag.min()), 
        max(y_test_mag.max(), y_test_pred_mag.max())]
ax.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Magnitude Réelle (m/s)')
ax.set_ylabel('Magnitude Prédite (m/s)')
ax.set_title('Magnitude: Prédiction vs Réalité')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("✓ Visualisations des prédictions générées")

## Part 9: Démonstration Interactive

Sélectionnez un scénario du test set et voyez la prédiction du modèle.

In [ ]:
# Sélectionner quelques scénarios intéressants
sample_indices = [0, len(X_test)//4, len(X_test)//2, 3*len(X_test)//4]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for plot_idx, sample_idx in enumerate(sample_indices):
    ax = axes[plot_idx]
    
    # Récupérer les données du scénario
    x_sample = X_test[sample_idx]
    y_true = y_test[sample_idx]
    y_pred = y_test_pred[sample_idx]
    
    # Extraire les positions et vitesses
    pos_sat = x_sample[0:3]
    vel_sat = x_sample[3:6]
    pos_deb = x_sample[6:9]
    vel_deb = x_sample[9:12]
    
    # Magnitudes
    mag_true = np.linalg.norm(y_true) * 1000
    mag_pred = np.linalg.norm(y_pred) * 1000
    error_pct = abs(mag_true - mag_pred) / mag_true * 100 if mag_true > 0 else 0
    
    # Distance initiale
    distance = np.linalg.norm(pos_sat - pos_deb)
    
    # Affichage
    ax.text(0.5, 0.95, f'Scénario {sample_idx}', 
            ha='center', va='top', fontsize=12, fontweight='bold',
            transform=ax.transAxes)
    
    text_str = f'''Distance satellite-débris: {distance:.2f} km
    
Magnitude Δv:
  Réel:      {mag_true:.4f} m/s
  Prédiction: {mag_pred:.4f} m/s
  Erreur:    {error_pct:.2f}%

Composantes (m/s):
           Réel      Prédiction  Erreur
  X:  {y_true[0]*1000:8.4f}    {y_pred[0]*1000:8.4f}   {(y_pred[0]-y_true[0])*1000:7.4f}
  Y:  {y_true[1]*1000:8.4f}    {y_pred[1]*1000:8.4f}   {(y_pred[1]-y_true[1])*1000:7.4f}
  Z:  {y_true[2]*1000:8.4f}    {y_pred[2]*1000:8.4f}   {(y_pred[2]-y_true[2])*1000:7.4f}'''
    
    ax.text(0.05, 0.85, text_str, 
            ha='left', va='top', fontsize=9, family='monospace',
            transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ Démonstration interactive affichée")

## Part 10: Importance des Features

Analysons l'importance de chaque feature pour le meilleur modèle.

In [ ]:
# Essayer d'obtenir l'importance des features si disponible
feature_names = ["Pos_Sat_X", "Pos_Sat_Y", "Pos_Sat_Z", 
                 "Vel_Sat_X", "Vel_Sat_Y", "Vel_Sat_Z",
                 "Pos_Deb_X", "Pos_Deb_Y", "Pos_Deb_Z",
                 "Vel_Deb_X", "Vel_Deb_Y", "Vel_Deb_Z"]

# Vérifier si le modèle a une méthode feature_importances
if hasattr(best_model.model, 'feature_importances_'):
    importances = best_model.model.feature_importances_
    
    # Trier par importance
    indices = np.argsort(importances)[::-1]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(indices)), importances[indices], color='steelblue', edgecolor='black')
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([feature_names[i] for i in indices])
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importance - {best_model_type}')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Feature importance affichée")
else:
    print("ℹ Le modèle ne fournit pas d'importance de features")
    print("  (MLP ne supporte pas nativement feature_importances_)")

## Part 11: Performance par Plage de Magnitude

Analysons la performance du modèle en fonction de la magnitude de la manœuvre.

In [ ]:
# Diviser les données en quartiles par magnitude
quartile_edges = np.percentile(y_test_mag, [25, 50, 75])
quartiles = np.digitize(y_test_mag, quartile_edges)

quartile_labels = ['Q1: 0-25%', 'Q2: 25-50%', 'Q3: 50-75%', 'Q4: 75-100%']
mae_by_quartile = []
rmse_by_quartile = []
r2_by_quartile = []

for q in range(4):
    mask = (quartiles == q)
    if mask.sum() > 0:
        y_true_q = y_test[mask]
        y_pred_q = y_test_pred[mask]
        
        mae = mean_absolute_error(y_true_q, y_pred_q) * 1000
        rmse = np.sqrt(mean_squared_error(y_true_q, y_pred_q)) * 1000
        r2 = r2_score(y_true_q, y_pred_q)
        
        mae_by_quartile.append(mae)
        rmse_by_quartile.append(rmse)
        r2_by_quartile.append(r2)

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# MAE par quartile
ax = axes[0]
ax.bar(quartile_labels, mae_by_quartile, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_ylabel('MAE (m/s)')
ax.set_title('Erreur MAE par Plage de Magnitude')
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(mae_by_quartile):
    ax.text(i, v, f'{v:.2f}', ha='center', va='bottom')

# RMSE par quartile
ax = axes[1]
ax.bar(quartile_labels, rmse_by_quartile, color='orange', alpha=0.7, edgecolor='black')
ax.set_ylabel('RMSE (m/s)')
ax.set_title('Erreur RMSE par Plage de Magnitude')
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(rmse_by_quartile):
    ax.text(i, v, f'{v:.2f}', ha='center', va='bottom')

# R² par quartile
ax = axes[2]
ax.bar(quartile_labels, r2_by_quartile, color='green', alpha=0.7, edgecolor='black')
ax.set_ylabel('R² Score')
ax.set_title('Score R² par Plage de Magnitude')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(r2_by_quartile):
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("✓ Analyse par plage de magnitude affichée")

## Part 12: Analyse de la Distance Satellite-Débris

Examinons comment la performance varie en fonction de la distance initiale.

In [ ]:
# Calculer les distances pour le test set
distances = np.linalg.norm(X_test[:, 0:3] - X_test[:, 6:9], axis=1)
errors = (y_test_mag - y_test_pred_mag)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distance vs Magnitude réelle
ax = axes[0]
scatter = ax.scatter(distances, y_test_mag, c=errors, cmap='RdYlGn_r', 
                     alpha=0.6, s=40, vmin=-2, vmax=2)
ax.set_xlabel('Distance Satellite-Débris (km)')
ax.set_ylabel('Magnitude Δv Réellement (m/s)')
ax.set_title('Distance vs Magnitude (coloré par erreur)')
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax, label='Erreur (m/s)')

# Erreur en fonction de la distance
ax = axes[1]
ax.scatter(distances, errors, alpha=0.6, s=40, color='purple')
ax.axhline(y=0, color='r', linestyle='--', lw=2)
ax.set_xlabel('Distance Satellite-Débris (km)')
ax.set_ylabel('Erreur de Prédiction (m/s)')
ax.set_title('Erreur vs Distance')
ax.grid(True, alpha=0.3)

# Ajouter une courbe de tendance
z = np.polyfit(distances, errors, 2)
p = np.poly1d(z)
distances_sorted = np.sort(distances)
ax.plot(distances_sorted, p(distances_sorted), "r-", linewidth=2, label='Tendance')
ax.legend()

plt.tight_layout()
plt.show()

print("✓ Analyse de distance affichée")

## Part 13: Conclusions

Résumé des résultats et performances.

In [ ]:
print("=" * 70)
print("RÉSUMÉ DES RÉSULTATS")
print("=" * 70)

print(f"\n✓ Meilleur modèle: {best_model_type.upper()}")
print(f"\nPerformances (Test Set):")
print(f"  • R² Score:           {test_r2:.4f}")
print(f"  • MAE:                {test_mae*1000:.4f} m/s")
print(f"  • RMSE:               {test_rmse*1000:.4f} m/s")

print(f"\nCaractéristiques du dataset:")
print(f"  • Nombre total de scénarios: {len(X)} ")
print(f"  • Nombre de features: {X.shape[1]}")
print(f"  • Nombre de targets (Δv composantes): {y.shape[1]}")

print(f"\nRésultats par quartile de magnitude:")
for i, label in enumerate(quartile_labels):
    print(f"  {label}:")
    print(f"    - MAE:  {mae_by_quartile[i]:.4f} m/s")
    print(f"    - RMSE: {rmse_by_quartile[i]:.4f} m/s")
    print(f"    - R²:   {r2_by_quartile[i]:.4f}")

print(f"\nStats erreur de prédiction (magnitude):")
print(f"  • Min:    {residuals_mag.min():+.4f} m/s")
print(f"  • Mean:   {residuals_mag.mean():+.4f} m/s")
print(f"  • Median: {np.median(residuals_mag):+.4f} m/s")
print(f"  • Max:    {residuals_mag.max():+.4f} m/s")
print(f"  • Std:    {residuals_mag.std():.4f} m/s")

print("\n" + "=" * 70)
print("Prochaines étapes:")
print("=" * 70)
print("""
1. Augmenter le nombre de scénarios (5000+) pour améliorer la généralisation
2. Ajuster les hyperparamètres du meilleur modèle
3. Valider sur des scénarios réalistes
4. Intégrer le modèle dans un système temps réel
5. Comparer avec d'autres approches (optimisation classique)

Merci d'avoir exploré ce notebook! 🚀
""")

## Appendix: Ressources et Documentation

Pour en savoir plus sur le projet:

In [ ]:
print("""
📚 RESSOURCES ET DOCUMENTATION
===============================================================================

Fichiers clés du projet:
  • src/orbital_mechanics.py   - Modèles de mécanique orbitale
  • src/dataset_generator.py   - Génération d'ensembles d'entraînement
  • src/ml_model.py            - Modèles et comparaison ML
  • src/optimizer.py           - Optimisation classique
  • src/sensor_model.py        - Modèles de capteurs optiques
  • src/collision_risk.py      - Calcul du risque de collision
  
Commandes principales:
  • python main.py             - Exécuter le pipeline complet
  • python main.py --n-scenarios 5000 - Générer plus de données
  
Structure des données:
  • X (features):  [position_sat(3), velocity_sat(3), position_deb(3), velocity_deb(3)]
  • y (labels):    [delta_v_x, delta_v_y, delta_v_z]
  
Modèles disponibles:
  • 'random_forest'      - Forêts aléatoires (rapide, robuste)
  • 'gradient_boosting'  - Gradient boosting (précis)
  • 'mlp'                - Réseau de neurones (flexible)
  
Métriques de performance:
  • R²:   Coefficient de détermination (0-1, plus haut = mieux)
  • MAE:  Erreur absolue moyenne (en km/s)
  • RMSE: RMS error (en km/s)

Contacts et support:
  📧 mohsine.essat@gmail.com
  🌐 https://github.com/orbital_avoidance_ml
  
===============================================================================
""")

print("\n✨ Notebook de démonstration complété avec succès!")

# Évitement Orbital avec Machine Learning

## Démonstration Interactive

Bienvenue dans le notebook de démonstration pour le projet d'évitement orbital assisté par ML!

Ce notebook présente:
- 🛰️ **Génération de scénarios** d'évitement orbital
- 🤖 **Entraînement de modèles** ML pour prédire les manœuvres optimales
- 📊 **Comparaison** de différentes approches (Random Forest, Gradient Boosting, MLP)
- 📈 **Visualisation** des résultats et performances

Amusez-vous à explorer!